# Droid — an autonomous software engineering agent (Factory-style)

[Factory](https://factory.ai) builds **Droids**: autonomous software engineering
agents that take a ticket, explore a codebase, form a plan, and iterate until
the test suite is green — then hand back a structured report of everything they did.

```
ticket ──▶ explore → reproduce → fix → verify (repeat) ──▶ structured run report
                  ↕               ↕
            filesystem tools   shell runner
            (code search +     (test suite,
             patch editing)     linter, etc.)
```

**SDK primitives this notebook showcases:**
- `FileSystemToolConfig` + `ReadTextTool`, `WriteTextTool`, `ListDirTool` — the SDK's built-in filesystem layer
- `GlobTool`, `GrepTool` — built-in code-search tools, scoped to a root directory
- `PatchTool` — built-in exact-string patch editor (safer than full-file rewrites)
- A custom `@tool` for running shell commands (the one thing the built-ins don't provide)
- `TokenBudgetMiddleware`, `CostBudgetMiddleware`, `RuntimeLimitMiddleware`, `LoopDetectionMiddleware` — budget and safety guards
- `TraceOption.continual(ActionTrace)` — the structured run report

> **Before running:** copy `.env.example` to `.env` at the repo root and add your provider key.

## 1. Environment & constants

| Constant | What it controls |
|---|---|
| `PROVIDER` / `MODEL` | The LLM that powers the droid. Default: OpenAI `gpt-4.1`. |
| `WORKSPACE` | The directory the droid can read and write. All filesystem and code-search tools are scoped here — the droid cannot touch anything outside it. |
| `MAX_ITERATIONS` | Hard cap on agent loop iterations. The middleware enforces this independently of the agent's own `max_iterations` — two layers of the same limit, belt and suspenders. |
| `TOKEN_BUDGET` | Maximum tokens the droid may spend per run. `TokenBudgetMiddleware` aborts if this is exceeded. |
| `COST_BUDGET_USD` | Maximum dollar cost (input + output tokens combined) per run. `CostBudgetMiddleware` aborts if this is exceeded. |

In [ ]:
%pip install -q vidbyte-sdk python-dotenv

import os
import subprocess
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()
load_dotenv("../../.env")

PROVIDER      = os.getenv("VIDBYTE_COOKBOOK_PROVIDER", "openai")
MODEL         = os.getenv("VIDBYTE_COOKBOOK_MODEL", "gpt-4.1")
WORKSPACE     = Path("workspace").resolve()   # droid is jailed to this dir
MAX_ITERATIONS = 30
TOKEN_BUDGET   = 60_000
COST_BUDGET_USD = 1.00

assert os.getenv("OPENAI_API_KEY") or os.getenv("ANTHROPIC_API_KEY"), \
    "Set OPENAI_API_KEY (or ANTHROPIC_API_KEY + overrides) in .env first."

WORKSPACE.mkdir(exist_ok=True)
print(f"Workspace: {WORKSPACE}")

## 2. System prompt

The droid's system prompt encodes its **operating procedure** — the sequence
of steps it must follow for every ticket. This is where the "never fix before
you reproduce, never report success before you verify" discipline lives.

Three things worth adapting if you use this for your own codebase:

- **Step 3 (FIX):** add language-specific style rules ("follow PEP 8",
  "preserve existing types")
- **Scope constraints:** add "do not touch migrations", "do not rename public
  APIs", etc.
- **Verification:** replace the generic `python -m unittest` with your actual
  test command (`pytest`, `cargo test`, `go test ./...`)

In [ ]:
SYSTEM_PROMPT = """You are a software engineering droid.

Operating procedure — follow every step, in order:

1. EXPLORE: list the repository files, then read the ones relevant to the ticket.
2. REPRODUCE: run the test suite and confirm you can see the exact failure
   described in the ticket. Never attempt a fix before you have reproduced it.
3. FIX: make the smallest correct change. Preserve existing style, naming, and
   public interfaces. Do not refactor code that is not directly relevant.
4. VERIFY: re-run the full test suite. If any test fails, iterate from step 3.
5. REPORT: once the suite is fully green, summarize the root cause, the exact
   change you made, and the passing test run.

Never claim success without a passing verification run in this session."""

## 3. Tools

The droid's hands. We use SDK built-ins wherever they exist — they handle path
scoping, permission enforcement, and error normalization for us.

| Tool | Source | What it does |
|---|---|---|
| `ListDirTool` | SDK filesystem | List directory contents, scoped to `WORKSPACE` |
| `ReadTextTool` | SDK filesystem | Read a file as text |
| `WriteTextTool` | SDK filesystem | Overwrite a file with new content |
| `GlobTool` | SDK code search | Find files by glob pattern (`**/*.py`, etc.) |
| `GrepTool` | SDK code search | Search file contents by regex, returns matching lines + locations |
| `PatchTool` | SDK editing | Exact-string patch edit — safer than full-file replacement |
| `run_shell` | Custom `@tool` | Run one shell command from the workspace root (the one thing built-ins don't provide) |

`FileSystemToolConfig` is the SDK's scoping layer — every filesystem and code-search
tool takes one and refuses to operate outside the configured `root`.

In [ ]:
from vidbyte import tool
from vidbyte.lib.dataclasses.filesystem import FileSystemToolConfig
from vidbyte.tools.builtins.code_search import GlobTool, GrepTool
from vidbyte.tools.builtins.editing import PatchTool
from vidbyte.tools.filesystem import ListDirTool, ReadTextTool, WriteTextTool

# One shared config — all filesystem tools are scoped to WORKSPACE.
# allow_write=True is required for WriteTextTool and PatchTool.
fs_cfg = FileSystemToolConfig(root=WORKSPACE, allow_write=True)

list_dir  = ListDirTool(fs_cfg)
read_file = ReadTextTool(fs_cfg)
write_file = WriteTextTool(fs_cfg)
glob      = GlobTool(root_dir=WORKSPACE)
grep      = GrepTool(root_dir=WORKSPACE)
patch     = PatchTool(root_dir=WORKSPACE)


@tool
def run_shell(command: str) -> dict:
    """Run a shell command from the workspace root (e.g. 'python -m unittest -v',
    'python -m pytest', 'black --check .').  Returns exit_code, stdout, and
    stderr — each truncated to 4 000 chars so the context window stays bounded."""
    proc = subprocess.run(
        command, shell=True, cwd=WORKSPACE,
        capture_output=True, text=True, timeout=60,
    )
    return {
        "exit_code": proc.returncode,
        "stdout": proc.stdout[-4_000:],
        "stderr": proc.stderr[-4_000:],
    }

## 4. Middleware

Middleware is **deterministic runtime policy that runs inside the agent loop
but is never visible to the model**. It can observe, slow, block, or abort
the run based on hard thresholds — regardless of what the model decides.

This is the layer that makes an autonomous agent safe to run unattended:

| Middleware | Why it matters for a coding droid |
|---|---|
| `TokenBudgetMiddleware` | Aborts if cumulative token usage exceeds `TOKEN_BUDGET`. Prevents runaway loops from draining your API quota. |
| `CostBudgetMiddleware` | Aborts if estimated dollar cost exceeds `COST_BUDGET_USD`. A second financial safety net on top of the token cap. |
| `RuntimeLimitMiddleware` | Aborts if the droid exceeds `MAX_ITERATIONS` iterations or 3 minutes of wall time — independent of `max_iterations` on the agent. |
| `LoopDetectionMiddleware` | Detects and aborts repetitive reasoning loops (same tool, same args, called 3+ times). Catches the "stuck in an exploration spiral" failure mode. |
| `ModelRetryMiddleware` | Automatically retries on transient provider errors (timeouts, rate limits) up to 3 times with backoff. |

In [ ]:
from vidbyte.middleware import (
    CostBudgetMiddleware,
    LoopDetectionMiddleware,
    ModelRetryMiddleware,
    RuntimeLimitMiddleware,
    TokenBudgetMiddleware,
)

middleware = [
    TokenBudgetMiddleware(max_tokens=TOKEN_BUDGET),
    CostBudgetMiddleware(max_cost_usd=COST_BUDGET_USD),
    RuntimeLimitMiddleware(max_iterations=MAX_ITERATIONS, max_elapsed_seconds=180.0),
    LoopDetectionMiddleware(max_repeats=3),
    ModelRetryMiddleware(max_retries=3),
]

## 5. Constructing the agent

**Requirements for a coding droid:**

- `PermissionPolicy.allow_all()` — the droid needs `WRITE` (to edit files) and
  `EXECUTE` (to run shell commands). The default policy denies both, so we
  explicitly grant them. In a production system you would scope this more
  tightly — allow writes only to `src/`, deny shell commands that touch the
  network, etc.
- `TraceOption.continual(ActionTrace)` — while the droid works, a dedicated
  background agent maintains a running structured artifact (goal, actions taken,
  mistakes, current status). This is the minimal version of the session report a
  real Droid hands back with its PR. It arrives on `reply.metadata["trace"]`.
- `max_iterations=MAX_ITERATIONS` — a per-agent limit that complements the
  middleware cap. The first one to fire wins.

In [ ]:
from vidbyte import BaseAgent, TraceOption
from vidbyte.tools.security import PermissionPolicy
from vidbyte.trace.continual import ActionTrace

droid = BaseAgent(
    name="droid",
    system_prompt=SYSTEM_PROMPT,
    provider=PROVIDER,
    model_name=MODEL,
    tools=[list_dir, read_file, write_file, glob, grep, patch, run_shell],
    middleware=middleware,
    permission_policy=PermissionPolicy.allow_all(),
    max_iterations=MAX_ITERATIONS,
    trace_option=TraceOption.continual(ActionTrace, every_n_iterations=3),
)

print("Droid ready. Tools:", [s.name for s in droid.tool_specs()])

## 6. Running the droid

Call `droid.run(<ticket>)` — the droid explores, reproduces, fixes, verifies,
and returns a natural-language report in `reply.content` plus a structured
trace artifact in `reply.metadata["trace"]`.

**Example tickets you can try:**

```
# Bug fix
"The apply_discount function treats a 0-100 percent as a 0-1 fraction.
 A 10% discount on $100 should give $90, not -$900. Fix it and make
 the test suite pass."

# Feature addition
"Add a validate_cart(cart) function that raises ValueError if any item
 has a negative price or zero quantity. Add tests for it."

# Refactor
"The cart_total function is hard to read. Extract the discount logic
 into a helper and add a docstring explaining the expected input ranges."
```

In the cell below, we seed a small demo repo first, then hand the droid a
real bug ticket. **Replace the TICKET string with any of the examples above**
to see how the droid adapts.

In [ ]:
import shutil, textwrap

# ── seed a tiny demo repo with a real bug ──────────────────────────
shutil.rmtree(WORKSPACE, ignore_errors=True)
WORKSPACE.mkdir()

(WORKSPACE / "inventory.py").write_text(textwrap.dedent('''
    def apply_discount(price, percent):
        # Apply a percentage discount (0-100) to a price.
        return price - price * percent   # bug: treats percent as fraction

    def cart_total(cart, discount_percent=0):
        subtotal = sum(item["price"] * item["qty"] for item in cart)
        from inventory import apply_discount
        return round(apply_discount(subtotal, discount_percent), 2)
''').lstrip(), encoding="utf-8")

(WORKSPACE / "test_inventory.py").write_text(textwrap.dedent('''
    import unittest
    from inventory import apply_discount, cart_total

    class TestInventory(unittest.TestCase):
        def test_ten_percent(self): self.assertEqual(apply_discount(100.0, 10), 90.0)
        def test_no_discount(self):  self.assertEqual(apply_discount(50.0,  0), 50.0)
        def test_cart(self):
            cart = [{"price": 25.0, "qty": 2}, {"price": 50.0, "qty": 1}]
            self.assertEqual(cart_total(cart, 10), 90.0)
''').lstrip(), encoding="utf-8")

TICKET = (
    "TICKET-1042: Discounts produce absurd totals\n\n"
    "Customer reports: a $100 cart with a 10% discount shows a total of -900.0.\n"
    "Expected: 90.0. The discount percent is a 0-100 value, not a fraction.\n\n"
    "Acceptance criteria: python -m unittest passes with exit code 0."
)

reply = droid.run(TICKET)
print(reply.content)

## 7. Example output

### Run report (`reply.content`)

```
Root cause: `apply_discount` multiplied `price * percent` directly, treating
the 0-100 percent as a 0-1 fraction. The fix divides by 100 before multiplying.

Change made: `inventory.py` line 3
  Before: return price - price * percent
  After:  return price - price * (percent / 100)

Verification: `python -m unittest -v` — 3 tests, 0 failures, exit code 0. ✓
```

### Structured trace (`reply.metadata["trace"]`)

```json
{
  "goal": "Fix apply_discount to treat percent as a 0-100 value.",
  "actions_taken": [
    "Listed workspace files: inventory.py, test_inventory.py",
    "Read inventory.py — found bug on line 3",
    "Reproduced failure: test_ten_percent FAIL (-900.0 != 90.0)",
    "Patched inventory.py: divided percent by 100",
    "Verified: 3/3 tests pass"
  ],
  "mistakes": [],
  "current_status": "All tests green. Fix complete."
}
```

In [ ]:
import json

# Inspect the structured trace the background agent maintained during the run
trace = reply.metadata.get("trace") or droid.last_trace
print(json.dumps(trace, indent=2) if trace else "(trace not yet available — run the agent first)")